<a href="https://colab.research.google.com/github/NoT-Serna/Juan_Serna_FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

1. **Freshness Significantly improves SEO performance**
The study found that updating older content can dramatically improve its performance. The study shows that the content that was more than 365 days old but had been refreshed within the previous 30 days showed a 3.2x increase in health score and 57x more impressions compared with the pre-refresh level

Question: How was the split done when testing whether the freshness improves SEO perforamance? On what timeframes was the content updated and test if that timepsan was enough to get to the result.

2. **Search Position has an impact on clicks**
The paper found that CTR decreases sharply as pages move down Google's rankings. Weighted CTR was 0.423% for the top 3 positions, compared with only 0.050% for pages ranking beyond position 50 with an 88% decrease

Question: Does the type of content of the page influence this claim? Perhaps there are cotent types that just naturally have a lower search position due to hrelevant it is to the user.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# ============================================================
# WEEK 6 — LOGISTIC REGRESSION
# RANDOM SPLIT vs TIME-AWARE SPLIT
#
# Objective:
# Predict whether content needs review in the NEXT month.
#
# RANDOM SPLIT:
#   April features  -> May outcome
#   Random 80/20 train/test split
#
# TIME-AWARE SPLIT:
#   April features  -> May outcome  [TRAIN]
#   May features    -> June outcome [TEST]
# ============================================================

import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

# Read the token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Register the Hugging Face secret
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

# Warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"


# ============================================================
# 1. EXPECTED CTR FROM WEEK 4
# ============================================================

expected_ctr = {
    "1-3": 0.019535,
    "3-5": 0.018284,
    "5-10": 0.008456,
    "10-20": 0.006893,
    "20+": 0.002238
}


# ============================================================
# 2. FUNCTION TO BUILD MONTHLY DATA
# ============================================================

def get_month_data(month):

    df = con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '{month}'
        GROUP BY
            client_hash_id,
            content_hash_id
    """).df()

    # --------------------------------------------------------
    # CTR
    # --------------------------------------------------------

    df["ctr"] = np.where(
        df["impressions"] > 0,
        df["clicks"] / df["impressions"],
        0
    )

    # --------------------------------------------------------
    # Position bucket
    # --------------------------------------------------------

    df["position_bucket"] = pd.cut(
        df["avg_position"],
        bins=[0, 3, 5, 10, 20, float("inf")],
        labels=[
            "1-3",
            "3-5",
            "5-10",
            "10-20",
            "20+"
        ],
        include_lowest=True
    )

    # --------------------------------------------------------
    # Expected CTR
    # --------------------------------------------------------

    df["expected_ctr"] = (
        df["position_bucket"]
        .astype(str)
        .map(expected_ctr)
    )

    # --------------------------------------------------------
    # CTR gap
    # --------------------------------------------------------

    df["ctr_gap"] = (
        df["expected_ctr"] - df["ctr"]
    )

    # --------------------------------------------------------
    # Needs-review label
    # --------------------------------------------------------

    df["needs_review"] = (
        (df["ctr_gap"] > 0) &
        (df["impressions"] >= 100)
    ).astype(int)

    return df


# ============================================================
# 3. LOAD APRIL, MAY AND JUNE
# ============================================================

april = get_month_data("2025-04")
may = get_month_data("2025-05")
june = get_month_data("2025-06")


print("April rows:", len(april))
print("May rows:", len(may))
print("June rows:", len(june))


# ============================================================
# 4. FEATURES
# ============================================================

features = [
    "impressions",
    "avg_position",
    "ctr"
]


# ============================================================
# 5. CREATE APRIL → MAY DATASET
#
# APRIL = FEATURES
# MAY    = TARGET
#
# IMPORTANT:
# We rename May's target BEFORE merging.
# This prevents the needs_review column collision.
# ============================================================

may_target = may[
    [
        "client_hash_id",
        "content_hash_id",
        "needs_review"
    ]
].copy()

may_target = may_target.rename(
    columns={
        "needs_review": "target"
    }
)


# Keep only the April features we need
april_features = april[
    [
        "client_hash_id",
        "content_hash_id"
    ] + features
].copy()


# Merge April features with May target
april_to_may = april_features.merge(
    may_target,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)


print("\nApril → May dataset:")
print("Rows:", len(april_to_may))
print(
    "Positive cases:",
    april_to_may["target"].sum()
)

print(
    "Negative cases:",
    (april_to_may["target"] == 0).sum()
)


# ============================================================
# 6. CREATE MAY → JUNE DATASET
#
# MAY    = FEATURES
# JUNE   = TARGET
#
# This is the genuine future test set.
# ============================================================

june_target = june[
    [
        "client_hash_id",
        "content_hash_id",
        "needs_review"
    ]
].copy()

june_target = june_target.rename(
    columns={
        "needs_review": "target"
    }
)


# Keep only May features
may_features = may[
    [
        "client_hash_id",
        "content_hash_id"
    ] + features
].copy()


# Merge May features with June target
may_to_june = may_features.merge(
    june_target,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)


print("\nMay → June dataset:")
print("Rows:", len(may_to_june))
print(
    "Positive cases:",
    may_to_june["target"].sum()
)

print(
    "Negative cases:",
    (may_to_june["target"] == 0).sum()
)


# ============================================================
# 7. SANITY CHECK
# ============================================================

print("\nTarget distribution:")
print(
    pd.DataFrame({
        "Dataset": [
            "April → May",
            "May → June"
        ],
        "Negative": [
            (april_to_may["target"] == 0).sum(),
            (may_to_june["target"] == 0).sum()
        ],
        "Positive": [
            (april_to_may["target"] == 1).sum(),
            (may_to_june["target"] == 1).sum()
        ]
    })
)


# ============================================================
# 8. CREATE MODEL
# ============================================================

def create_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "logistic",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])


# ============================================================
# 9. RANDOM SPLIT
#
# APRIL FEATURES → MAY OUTCOME
#
# 80% TRAIN
# 20% TEST
# ============================================================

X_random = april_to_may[features]
y_random = april_to_may["target"]


X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X_random,
        y_random,
        test_size=0.20,
        random_state=42,
        stratify=y_random
    )
)


# Create model
random_model = create_model()


# Train
random_model.fit(
    X_train_random,
    y_train_random
)


# Predict test set
y_pred_random = random_model.predict(
    X_test_random
)


# F1
random_f1 = f1_score(
    y_test_random,
    y_pred_random
)


# ============================================================
# 10. RANDOM SPLIT RESULTS
# ============================================================

print("\n")
print("=" * 60)
print("RANDOM SPLIT — APRIL → MAY")
print("=" * 60)

print(
    "Training observations:",
    len(X_train_random)
)

print(
    "Testing observations :",
    len(X_test_random)
)

print(
    "\nF1 Score:",
    round(random_f1, 4)
)

print("\nClassification Report:")

print(
    classification_report(
        y_test_random,
        y_pred_random,
        digits=4
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test_random,
        y_pred_random
    )
)


# ============================================================
# 11. TIME-AWARE SPLIT
#
# TRAIN:
#   April features → May outcome
#
# TEST:
#   May features → June outcome
#
# IMPORTANT:
# The model does NOT use May or June outcomes during training.
# ============================================================

X_train_time = april_to_may[features]
y_train_time = april_to_may["target"]


X_test_time = may_to_june[features]
y_test_time = may_to_june["target"]


# Create fresh model
time_model = create_model()


# Train ONLY on April → May
time_model.fit(
    X_train_time,
    y_train_time
)


# Predict May → June
y_pred_time = time_model.predict(
    X_test_time
)


# F1
time_f1 = f1_score(
    y_test_time,
    y_pred_time
)


# ============================================================
# 12. TIME-AWARE RESULTS
# ============================================================

print("\n")
print("=" * 60)
print("TIME-AWARE SPLIT — APRIL → MAY → JUNE")
print("=" * 60)

print(
    "Training observations (April → May):",
    len(X_train_time)
)

print(
    "Testing observations (May → June):",
    len(X_test_time)
)

print(
    "\nF1 Score:",
    round(time_f1, 4)
)

print("\nClassification Report:")

print(
    classification_report(
        y_test_time,
        y_pred_time,
        digits=4
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test_time,
        y_pred_time
    )
)


# ============================================================
# 13. FINAL COMPARISON
# ============================================================

comparison = pd.DataFrame({

    "Evaluation": [
        "Random Split",
        "Time-Aware Split"
    ],

    "Training Features": [
        "April",
        "April"
    ],

    "Testing Features": [
        "April",
        "May"
    ],

    "Target Outcome": [
        "May",
        "June"
    ],

    "F1": [
        random_f1,
        time_f1
    ]
})


print("\n")
print("=" * 60)
print("RANDOM vs TIME-AWARE")
print("=" * 60)

display(comparison)


# ============================================================
# 14. PERFORMANCE DROP
# ============================================================

f1_difference = random_f1 - time_f1


if random_f1 != 0:

    percentage_drop = (
        (random_f1 - time_f1) / random_f1
    ) * 100

else:

    percentage_drop = np.nan


print(
    "\nF1 difference:",
    round(f1_difference, 4)
)

print(
    "Percentage decrease:",
    round(percentage_drop, 2),
    "%"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April rows: 13046
May rows: 14887
June rows: 16399

April → May dataset:
Rows: 12616
Positive cases: 7014
Negative cases: 5602

May → June dataset:
Rows: 13948
Positive cases: 7081
Negative cases: 6867

Target distribution:
       Dataset  Negative  Positive
0  April → May      5602      7014
1   May → June      6867      7081


RANDOM SPLIT — APRIL → MAY
Training observations: 10092
Testing observations : 2524

F1 Score: 0.7092

Classification Report:
              precision    recall  f1-score   support

           0     0.6326    0.4639    0.5353      1121
           1     0.6469    0.7847    0.7092      1403

    accuracy                         0.6422      2524
   macro avg     0.6397    0.6243    0.6222      2524
weighted avg     0.6405    0.6422    0.6319      2524

Confusion Matrix:
[[ 520  601]
 [ 302 1101]]


TIME-AWARE SPLIT — APRIL → MAY → JUNE
Training observations (April → May): 12616
Testing observations (May → June): 13948

F1 Score: 0.7157

Classification Report:
     

,Evaluation,Training Features,Testing Features,Target Outcome,F1
0,Random Split,April,April,May,0.709179
1,Time-Aware Split,April,May,June,0.715738



F1 difference: -0.0066
Percentage decrease: -0.92 %


  The logistic regression model achieved an F1 socre of 0.7092 under the random 80/20 split and 0.7157 under the time-aware evaluation. The model maintained and slightly improved its F1 score when evaluated on future data. The difference of 0.0066 F1 points suggests that there is no substantial temporal degradation in this experiment. The time-aware model achieved 87% recall for content requiring review, indicating that it successfully identifies most positive cases, although its precision of 60.8% means that a considerable number of false positives would require manual review.

  Finally, the results suggest that the relationship between impressions,average position, CTR m and the rule defined needs_review outcome is reasonably stable across the April-June period.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# ============================================================
# WEEK 6 — LEAKAGE AUDIT
# ============================================================

print("\n")
print("=" * 60)
print("LEAKAGE AUDIT")
print("=" * 60)


# ------------------------------------------------------------
# TEST 1 — Check model features
# ------------------------------------------------------------

forbidden_features = [
    "needs_review",
    "expected_ctr",
    "ctr_gap",
    "position_bucket"
]

print("\n[1] FEATURE LEAKAGE CHECK")

leaked_features = [
    col for col in features
    if col in forbidden_features
]

if len(leaked_features) == 0:
    print("PASS — No target-derived features used.")
else:
    print("FAIL — Potential leakage:")
    print(leaked_features)


# ------------------------------------------------------------
# TEST 2 — Check that target is not in X
# ------------------------------------------------------------

print("\n[2] TARGET IN FEATURES CHECK")

if "target" not in X_train_time.columns:
    print("PASS — Target is not included in training features.")
else:
    print("FAIL — Target is included in training features!")


# ------------------------------------------------------------
# TEST 3 — Check time-aware periods
# ------------------------------------------------------------

print("\n[3] TIME-AWARE PERIOD CHECK")

print("Training features: April")
print("Training target:   May")
print("Testing features:  May")
print("Testing target:    June")

print("PASS — Temporal order is preserved.")


# ------------------------------------------------------------
# TEST 4 — Check preprocessing
# ------------------------------------------------------------

print("\n[4] PREPROCESSING CHECK")

print(
    "Imputer and scaler are inside the Pipeline."
)

print(
    "PASS — Preprocessing is fitted only on training data."
)


# ------------------------------------------------------------
# TEST 5 — Check IDs are not features
# ------------------------------------------------------------

print("\n[5] ID FEATURE CHECK")

id_columns = [
    "client_hash_id",
    "content_hash_id"
]

id_leakage = [
    col for col in features
    if col in id_columns
]

if len(id_leakage) == 0:
    print("PASS — IDs are not used as model features.")
else:
    print("FAIL — IDs are being used as features:")
    print(id_leakage)


# ------------------------------------------------------------
# TEST 6 — Check future target is not in training data
# ------------------------------------------------------------

print("\n[6] FUTURE TARGET CHECK")

train_columns = set(april_to_may.columns)

future_target_columns = {
    "target",
    "needs_review"
}

# target itself is expected in the supervised dataset,
# but must NOT be in X.

train_feature_columns = set(X_train_time.columns)

unexpected = train_feature_columns.intersection(
    future_target_columns
)

if len(unexpected) == 0:
    print(
        "PASS — Future target is not present in training features."
    )
else:
    print(
        "FAIL — Future target found in training features:"
    )
    print(unexpected)


# ------------------------------------------------------------
# TEST 7 — Check feature consistency
# ------------------------------------------------------------

print("\n[7] TRAIN / TEST FEATURE CONSISTENCY")

if list(X_train_time.columns) == list(X_test_time.columns):

    print(
        "PASS — Training and testing use the same features."
    )

else:

    print(
        "FAIL — Training and testing features differ."
    )

    print(
        "Train:",
        list(X_train_time.columns)
    )

    print(
        "Test:",
        list(X_test_time.columns)
    )


# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LEAKAGE AUDIT COMPLETE")
print("=" * 60)

if (
    len(leaked_features) == 0
    and "target" not in X_train_time.columns
    and len(id_leakage) == 0
    and len(unexpected) == 0
    and list(X_train_time.columns) == list(X_test_time.columns)
):

    print("OVERALL RESULT: PASS")
    print("No direct target or temporal leakage detected.")

else:

    print("OVERALL RESULT: REVIEW REQUIRED")
    print("Potential leakage was detected.")




LEAKAGE AUDIT

[1] FEATURE LEAKAGE CHECK
PASS — No target-derived features used.

[2] TARGET IN FEATURES CHECK
PASS — Target is not included in training features.

[3] TIME-AWARE PERIOD CHECK
Training features: April
Training target:   May
Testing features:  May
Testing target:    June
PASS — Temporal order is preserved.

[4] PREPROCESSING CHECK
Imputer and scaler are inside the Pipeline.
PASS — Preprocessing is fitted only on training data.

[5] ID FEATURE CHECK
PASS — IDs are not used as model features.

[6] FUTURE TARGET CHECK
PASS — Future target is not present in training features.

[7] TRAIN / TEST FEATURE CONSISTENCY
PASS — Training and testing use the same features.

LEAKAGE AUDIT COMPLETE
OVERALL RESULT: PASS
No direct target or temporal leakage detected.


This code audits the model on different aspects:
1. Direct target leakage
2. Future-feature leakage
3. Future-target leakage
4. Preprocessing leakage:
5. ID leakage
6. Temporal Ordering

The leakage audit found no direcvt target,future-feature,preprocessing, or identifier leakage. The time-aware evaluation preserves the chronological structure of the predcition task, where April datra are used to train the model on May outcomes, while May outcomes predcit the month of June outcomes. The feature preprocessing is performed within the modeling pipeline, ensuring that imputation and scaling parameters are learned exclusively from the training data. The only limitation is that the random split may not provide fully independent observations when multiple pieces of content belong to the same client.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Bold Claim**
The Logistic Regression model can identify which content needs SEO optimization better than the Week 4 baseline rule

**Safe Claim**
The Logistic Regression model measured a different F1 score than the Week-4 baseline when evaluated on the future rule-defined review condition. This provide directional evidence about how well April features generalize to May-June and can support prioritization decisions, although it dones not establish that the model causes better SEO performance or identifies actual optimizaction opportunites where the discretion of manual revision would still be viable.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.